### E agora - Semana 3 Dia 3

## AutoGen Core

Algo um pouco diferente.

Isso é agnóstico em relação ao framework de agentes subjacente

Você pode usar o AutoGen AgentChat ou outra solução; é um framework de interação entre agentes.

Desse ponto de vista, ele se posiciona de forma semelhante ao LangGraph.

### O princípio fundamental

Autogen Core desacopla a lógica de um agente de como as mensagens são entregues.  
O framework fornece uma infraestrutura de comunicação, junto com o ciclo de vida dos agentes, e eles são responsáveis pelo próprio trabalho.

A infraestrutura de comunicação é chamada de Runtime.

Existem 2 tipos: **Standalone** e **Distributed**.

Hoje usaremos um runtime standalone: o **SingleThreadedAgentRuntime**, uma implementação local incorporada de runtime de agentes.

Amanhã veremos rapidamente um runtime distribuído.


In [9]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)


True

### Primeiro definimos o nosso objeto Message

Qualquer estrutura que quisermos para as mensagens no nosso framework de agentes.

In [10]:
# Vamos usar um exemplo simples!

@dataclass
class Message:
    content: str


### Agora definimos nosso agente

Uma subclasse de RoutedAgent.

Todo agente tem um **Agent ID** com 2 componentes:  
`agent.id.type` descreve o tipo de agente  
`agent.id.key` fornece seu identificador exclusivo

Qualquer método decorado com `@message_handler` terá a oportunidade de receber mensagens.


In [11]:
class SimpleAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Simple")

    @message_handler
    async def on_my_message(self, message: Message, ctx: MessageContext) -> Message:
        return Message(content=f"Aqui é {self.id.type}-{self.id.key}. Você disse '{message.content}' e eu discordo.")
        

### OK vamos criar um runtime Standalone e registrar nosso tipo de agente

In [12]:

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())

AgentType(type='simple_agent')

### Certo! Vamos iniciar um runtime e enviar uma mensagem

In [13]:
runtime.start()

In [14]:
agent_id = AgentId("simple_agent", "default")
response = await runtime.send_message(Message("Bem, olá!"), agent_id)
print(">>>", response.content)

>>> Aqui é simple_agent-default. Você disse 'Bem, olá!' e eu discordo.


In [15]:
await runtime.stop()
await runtime.close()

### OK agora vamos fazer algo mais interessante

Vamos usar um Assistant do AgentChat!

In [16]:

class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} recebeu a mensagem: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} respondeu: {reply}")
        return Message(content=reply)
    


In [17]:
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

AgentType(type='LLMAgent')

In [18]:
runtime.start()  # Inicie o processamento de mensagens em segundo plano.
response = await runtime.send_message(Message("Olá!"), AgentId("LLMAgent", "default"))
print(">>>", response.content)
response =  await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)
response = await runtime.send_message(Message(response.content), AgentId("LLMAgent", "default"))

LLMAgent recebeu a mensagem: Olá!
LLMAgent respondeu: Olá! Como posso ajudar você hoje?
>>> Olá! Como posso ajudar você hoje?
>>> Aqui é simple_agent-default. Você disse 'Olá! Como posso ajudar você hoje?' e eu discordo.
LLMAgent recebeu a mensagem: Aqui é simple_agent-default. Você disse 'Olá! Como posso ajudar você hoje?' e eu discordo.
LLMAgent respondeu: Entendo! Peço desculpas se a resposta não foi adequada. Como posso melhorar ou ajudar você da melhor forma?


In [19]:
await runtime.stop()
await runtime.close()

### OK agora vamos mostrar isso na prática — vamos fazer 3 agentes interagirem!

In [20]:
from autogen_ext.models.ollama import OllamaChatCompletionClient


class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [24]:
JUDGE = "Você está julgando uma partida de pedra, papel, tesoura. Os jogadores fizeram estas escolhas:\n"

class RockPaperScissorsAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "Você está jogando pedra, papel, tesoura. Responda apenas com uma palavra, escolhendo entre: pedra, papel ou tesoura."
        message = Message(content=instruction)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"Jogador 1: {response1.content}\nJogador 2: {response2.content}\n"
        judgement = f"{JUDGE}{result}Quem vence?"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)


In [25]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorsAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorsAgent("rock_paper_scissors"))
runtime.start()

In [26]:
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="vai")
response = await runtime.send_message(message, agent_id)
print(response.content)

Jogador 1: papel
Jogador 2: Papel.
A partida entre o Jogador 1 e o Jogador 2 resultou em um empate, pois ambos escolheram papel. 

TERMINATE


In [27]:
await runtime.stop()
await runtime.close()